# Lab 03: Reranker Pipeline

**Goal:** Build a two-stage retrieval pipeline that first retrieves a large candidate set (fast, approximate) then reranks it with a cross-encoder (slower, precise).

**What you'll build:**
1. Stage 1: Dense retrieval (bi-encoder) — fast, approximate
2. Stage 2: Cross-encoder reranker — slow, precise
3. Pipeline combining both stages
4. Comparison: bi-encoder only vs. reranked
5. Latency/cost trade-off measurement

**Key insight:** The reranker sees both query AND passage jointly (unlike bi-encoders which encode independently). This cross-attention is what makes cross-encoders more accurate but slower — they can't pre-compute document embeddings.

In [ ]:
!pip install sentence-transformers faiss-cpu google-genai anthropic openai python-dotenv --quiet

## 1. Sample Corpus (Technical RAG Domain)

In [ ]:
CORPUS = [
    {"id": "chunk_00", "text": "Chunking is the process of splitting documents into smaller pieces before embedding. The most common strategy is recursive character splitting with a chunk size of 512 tokens and 20% overlap."},
    {"id": "chunk_01", "text": "Fixed-size chunking splits text every N characters regardless of sentence boundaries. It is fast but often splits mid-sentence, which can hurt retrieval recall."},
    {"id": "chunk_02", "text": "Semantic chunking groups sentences by embedding similarity. When the cosine similarity between adjacent sentence embeddings drops below a threshold (e.g. 0.7), a chunk boundary is placed."},
    {"id": "embed_00", "text": "Text embeddings map text to fixed-size dense vectors in a high-dimensional space. The cosine similarity between two vectors measures how semantically related they are."},
    {"id": "embed_01", "text": "BGE (BAAI General Embeddings) models are strong open-source embedding models. BGE-M3 supports 100+ languages and long documents up to 8192 tokens."},
    {"id": "embed_02", "text": "Embedding models are evaluated on MTEB (Massive Text Embedding Benchmark). A score above 60 on MTEB retrieval tasks is considered production-grade for most use cases."},
    {"id": "rerank_00", "text": "Cross-encoder rerankers score each (query, document) pair jointly. Unlike bi-encoders, they cannot pre-compute document representations — they must process each pair at query time."},
    {"id": "rerank_01", "text": "The standard reranking pipeline: retrieve top-k candidates with a fast bi-encoder, then rerank with a cross-encoder. k=20-50 candidates for reranking is typical; k=3-5 final results."},
    {"id": "rerank_02", "text": "Popular cross-encoder rerankers include ms-marco-MiniLM-L-6-v2 (fast, accurate for general domains), cross-encoder/nli-deberta-v3-base (NLI tasks), and Cohere Rerank API (managed service)."},
    {"id": "rerank_03", "text": "Reranker latency scales linearly with k (the number of candidates). Reranking 20 candidates takes ~20ms on CPU; reranking 100 takes ~100ms. Keep k_rerank ≤ 50 for <200ms latency budgets."},
    {"id": "vector_00", "text": "HNSW (Hierarchical Navigable Small World) is the default ANN index for most vector databases. It achieves O(log N) query time and is the best choice for corpora under 100M vectors."},
    {"id": "vector_01", "text": "Pinecone, Weaviate, Qdrant, and Milvus are the leading vector databases. Pinecone is fully managed; Qdrant is self-hosted with native Rust performance; Milvus handles billion-scale."},
    {"id": "eval_00", "text": "NDCG@k (Normalized Discounted Cumulative Gain) measures retrieval quality considering rank order. A document retrieved at rank 1 counts more than rank 5. NDCG@5 is the standard evaluation metric."},
    {"id": "eval_01", "text": "Recall@k measures what fraction of relevant documents appear in the top-k results. It is the primary metric for RAG retrieval pipelines — missing a relevant document is worse than including an irrelevant one."},
    {"id": "hybrid_00", "text": "Hybrid search combines BM25 (keyword) and dense (semantic) retrieval. Results are merged with RRF. It outperforms either method alone on corpora with mixed exact-match and semantic queries."},
    {"id": "hybrid_01", "text": "RRF (Reciprocal Rank Fusion) merges ranked lists by summing 1/(k+rank) for each document across lists. The parameter k=60 is robust across tasks and is the standard default."},
]

print(f"Corpus: {len(CORPUS)} documents")

## 2. Stage 1: Bi-Encoder Retrieval (Fast)

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Bi-encoder: fast, encodes query and documents independently
bi_encoder = SentenceTransformer("BAAI/bge-small-en-v1.5")
print(f"Bi-encoder loaded: {bi_encoder.get_sentence_embedding_dimension()}d")

# Index all documents
corpus_texts = [doc["text"] for doc in CORPUS]
doc_embeddings = bi_encoder.encode(
    corpus_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings.astype(np.float32))
print(f"Index built: {index.ntotal} vectors")

In [ ]:
import time

def bi_encode_retrieve(query: str, k: int = 20) -> list[dict]:
    """Stage 1: Fast approximate retrieval using the bi-encoder."""
    query_emb = bi_encoder.encode(
        f"Represent this sentence for searching relevant passages: {query}",
        normalize_embeddings=True
    ).reshape(1, -1).astype(np.float32)

    scores, indices = index.search(query_emb, k)
    return [
        {
            "id":           CORPUS[idx]["id"],
            "text":         CORPUS[idx]["text"],
            "bi_score":     float(scores[0][rank]),
            "bi_rank":      rank + 1,
        }
        for rank, idx in enumerate(indices[0])
    ]

# Test
t0 = time.time()
candidates = bi_encode_retrieve("How does reranking improve RAG quality?")
print(f"Bi-encoder retrieval: {(time.time()-t0)*1000:.1f}ms — {len(candidates)} candidates")
for c in candidates[:5]:
    print(f"  [{c['bi_rank']}] {c['id']} ({c['bi_score']:.3f}): {c['text'][:60]}...")

## 3. Stage 2: Cross-Encoder Reranking (Precise)

In [ ]:
from sentence_transformers import CrossEncoder

# Cross-encoder: slow but sees query+document jointly (cross-attention)
# ms-marco-MiniLM-L-6-v2 is optimized for passage ranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded")

def rerank(query: str, candidates: list[dict], top_n: int = 5) -> list[dict]:
    """
    Stage 2: Rerank candidates using a cross-encoder.
    More accurate than bi-encoder because it sees query+document jointly.
    """
    # Build (query, passage) pairs
    pairs = [(query, c["text"]) for c in candidates]

    # Score all pairs in a single batch
    scores = cross_encoder.predict(pairs)

    # Sort by cross-encoder score
    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    return [
        {**doc, "ce_score": float(score), "ce_rank": rank + 1}
        for rank, (doc, score) in enumerate(ranked)
    ]


def two_stage_retrieve(query: str, k_candidates: int = 20, k_final: int = 5) -> list[dict]:
    """Full two-stage pipeline: bi-encoder → cross-encoder reranker."""
    candidates = bi_encode_retrieve(query, k=k_candidates)
    return rerank(query, candidates, top_n=k_final)


# Test
t0 = time.time()
results = two_stage_retrieve("How does reranking improve RAG quality?")
elapsed = (time.time() - t0) * 1000
print(f"Two-stage retrieval: {elapsed:.1f}ms\n")
for r in results:
    print(f"[{r['ce_rank']}] {r['id']} (bi_rank={r['bi_rank']}, ce_score={r['ce_score']:.3f})")
    print(f"    {r['text'][:80]}...\n")

## 4. Rank Change Analysis

Compare bi-encoder ranking vs. cross-encoder ranking to see where reranking makes a difference.

In [ ]:
def analyze_rank_changes(query: str, k_candidates: int = 15, k_final: int = 5):
    """Show how reranking changes the final top-k results."""
    candidates = bi_encode_retrieve(query, k=k_candidates)
    reranked   = rerank(query, candidates, top_n=k_final)

    print(f"Query: '{query}'")
    print(f"\nBi-encoder top-{k_final}:")
    for c in candidates[:k_final]:
        print(f"  [{c['bi_rank']}] {c['id']}: {c['text'][:65]}...")

    print(f"\nAfter reranking top-{k_final}:")
    for r in reranked:
        delta = r['bi_rank'] - r['ce_rank']
        arrow = f"↑{delta}" if delta > 0 else (f"↓{-delta}" if delta < 0 else "→")
        print(f"  [{r['ce_rank']}] {r['id']} ({arrow} from bi_rank={r['bi_rank']}): ce_score={r['ce_score']:.2f}")
        print(f"       {r['text'][:65]}...")


TEST_QUERIES = [
    "What is the difference between bi-encoder and cross-encoder?",
    "How many candidates should I use for reranking?",
    "Best way to evaluate retrieval quality",
]

for q in TEST_QUERIES:
    print("\n" + "="*70)
    analyze_rank_changes(q)

## 5. Latency vs. Accuracy Trade-off

In [ ]:
import statistics

def measure_latency(fn, query: str, n_runs: int = 10) -> dict:
    times = []
    for _ in range(n_runs):
        t0 = time.time()
        fn(query)
        times.append((time.time() - t0) * 1000)
    return {
        "mean_ms":   round(statistics.mean(times), 1),
        "p95_ms":    round(sorted(times)[int(0.95 * n_runs)], 1),
        "min_ms":    round(min(times), 1),
    }

query = "How does a reranker work?"

print("Latency comparison (10 runs each):")
print()

strategies = [
    ("Bi-encoder only (k=5)",      lambda q: bi_encode_retrieve(q, k=5)),
    ("Two-stage (k=20→5)",         lambda q: two_stage_retrieve(q, k_candidates=20, k_final=5)),
    ("Two-stage (k=50→5)",         lambda q: two_stage_retrieve(q, k_candidates=50, k_final=5)),
]

for name, fn in strategies:
    stats = measure_latency(fn, query)
    print(f"  {name:<35} mean={stats['mean_ms']:5.1f}ms  p95={stats['p95_ms']:5.1f}ms")

print()
print("Note: Cross-encoder latency is O(k_candidates) — keep k ≤ 50 for <200ms.")

## 6. End-to-End RAG with Reranker

In [ ]:
from ai_client import generate

def reranked_rag(query: str, k_candidates: int = 20, k_final: int = 4) -> str:
    """Full pipeline: retrieve 20 candidates → rerank to 4 → generate."""
    docs = two_stage_retrieve(query, k_candidates=k_candidates, k_final=k_final)
    context = "\n\n".join(f"[{d['id']}] {d['text']}" for d in docs)

    return generate(
        prompt=f"Context:\n{context}\n\nQuestion: {query}",
        system="Answer concisely using the provided context. Cite document IDs.",
        max_tokens=512,
    )


queries = [
    "How do I choose between a bi-encoder and cross-encoder for my RAG system?",
    "What k value should I use when reranking?",
]

for q in queries:
    print(f"Q: {q}")
    print(f"A: {reranked_rag(q)}")
    print()

## 7. Key Takeaways

| Stage | Model Type | Encodes | Latency | Accuracy | Use For |
|---|---|---|---|---|---|
| **Stage 1 (Retrieval)** | Bi-encoder | Query and document independently | Fast (<20ms) | Approximate | Get candidate set (k=20-50) |
| **Stage 2 (Reranking)** | Cross-encoder | Query + document jointly | Slow (O(k)) | Precise | Reorder to final top-k |

**When to add a reranker:**
- Latency budget allows >100ms (adds ~50-150ms for k=20)
- Corpus has domain-specific content where bi-encoders make ranking mistakes
- First-stage retrieval misses top-1 accuracy (add reranker before adding latency)

**Rule of thumb:** Start with k_candidates=20, k_final=5. Increase k_candidates if you notice relevant docs outside the initial 20.